<a href="https://colab.research.google.com/github/ntlcs/fiap-tech-challenge-fase-3/blob/main/03_Desafio_FIAP_IA_01_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tech Challenge - Fase 3

## Assistente Médico Inteligente com LLM, LangChain e LangGraph

### Pós-Tech FIAP - Inteligência Artificial para Developers

Este notebook contém a etapa de preparação, curadoria, anonimização e transformação dos dados utilizados no fine-tuning da LLM e na construção do assistente médico.

### Domínio clínico

A solução será inicialmente especializada no apoio à decisão clínica relacionada ao Diabetes Mellitus Tipo 2.

Serão utilizados dados sintéticos de pacientes e protocolos institucionais simulados, evitando o uso de informações pessoais reais.

### Objetivos desta etapa

- Criar dados clínicos sintéticos
- Estruturar protocolos hospitalares simulados
- Realizar limpeza e normalização dos dados
- Aplicar anonimização
- Remover duplicidades e registros inválidos
- Preparar o dataset para fine-tuning
- Gerar arquivos processados para as próximas etapas

In [ ]:
import sys
import platform

print("Python:", sys.version)
print("Sistema:", platform.platform())

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Sistema: Linux-6.6.122+-x86_64-with-glibc2.39


In [ ]:
!pip install -q pandas numpy datasets

In [ ]:
import json
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print(f"Seed definida: {SEED}")

Seed definida: 42


In [ ]:
BASE_DIR = Path("/content/fiap-tech-challenge-fase-3")

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
SYNTHETIC_DIR = BASE_DIR / "data" / "synthetic"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SYNTHETIC_DIR.mkdir(parents=True, exist_ok=True)

print(BASE_DIR)
print(RAW_DIR)
print(PROCESSED_DIR)
print(SYNTHETIC_DIR)

/content/fiap-tech-challenge-fase-3
/content/fiap-tech-challenge-fase-3/data/raw
/content/fiap-tech-challenge-fase-3/data/processed
/content/fiap-tech-challenge-fase-3/data/synthetic


## 1 Criação dos dados clínicos sintéticos

Para preservar a privacidade e evitar o uso de dados pessoais reais, serão utilizados registros clínicos sintéticos.

Os registros representam pacientes fictícios com diferentes níveis de controle do Diabetes Mellitus Tipo 2 e fatores clínicos associados.

Os dados sintéticos serão utilizados para demonstrar:

- consulta a prontuários;
- contextualização clínica;
- identificação de alterações laboratoriais;
- recuperação de protocolos;
- geração de alertas;
- apoio à decisão clínica.

In [ ]:
pacientes = [
    {
        "patient_id": "PAC001",
        "idade": 52,
        "sexo": "F",
        "diagnostico": "Diabetes Mellitus Tipo 2",
        "glicemia_mg_dl": 205,
        "hba1c_percentual": 9.2,
        "pressao_sistolica": 145,
        "pressao_diastolica": 95,
        "imc": 31.2,
        "creatinina_mg_dl": 1.0,
        "colesterol_total_mg_dl": 218,
        "exame_pendente": "Microalbuminúria"
    },
    {
        "patient_id": "PAC002",
        "idade": 46,
        "sexo": "M",
        "diagnostico": "Diabetes Mellitus Tipo 2",
        "glicemia_mg_dl": 118,
        "hba1c_percentual": 6.7,
        "pressao_sistolica": 128,
        "pressao_diastolica": 82,
        "imc": 27.4,
        "creatinina_mg_dl": 0.9,
        "colesterol_total_mg_dl": 185,
        "exame_pendente": None
    },
    {
        "patient_id": "PAC003",
        "idade": 63,
        "sexo": "F",
        "diagnostico": "Diabetes Mellitus Tipo 2",
        "glicemia_mg_dl": 172,
        "hba1c_percentual": 8.3,
        "pressao_sistolica": 138,
        "pressao_diastolica": 88,
        "imc": 29.8,
        "creatinina_mg_dl": 1.3,
        "colesterol_total_mg_dl": 201,
        "exame_pendente": "Fundo de olho"
    },
    {
        "patient_id": "PAC004",
        "idade": 39,
        "sexo": "M",
        "diagnostico": "Diabetes Mellitus Tipo 2",
        "glicemia_mg_dl": 96,
        "hba1c_percentual": 5.9,
        "pressao_sistolica": 122,
        "pressao_diastolica": 78,
        "imc": 25.6,
        "creatinina_mg_dl": 0.8,
        "colesterol_total_mg_dl": 174,
        "exame_pendente": None
    },
    {
        "patient_id": "PAC005",
        "idade": 58,
        "sexo": "F",
        "diagnostico": "Diabetes Mellitus Tipo 2",
        "glicemia_mg_dl": 238,
        "hba1c_percentual": 10.1,
        "pressao_sistolica": 154,
        "pressao_diastolica": 98,
        "imc": 33.5,
        "creatinina_mg_dl": 1.5,
        "colesterol_total_mg_dl": 243,
        "exame_pendente": "Avaliação renal"
    }
]

df_pacientes = pd.DataFrame(pacientes)

df_pacientes

,patient_id,idade,sexo,diagnostico,glicemia_mg_dl,hba1c_percentual,pressao_sistolica,pressao_diastolica,imc,creatinina_mg_dl,colesterol_total_mg_dl,exame_pendente
0,PAC001,52,F,Diabetes Mellitus Tipo 2,205,9.2,145,95,31.2,1.0,218,Microalbuminúria
1,PAC002,46,M,Diabetes Mellitus Tipo 2,118,6.7,128,82,27.4,0.9,185,None
2,PAC003,63,F,Diabetes Mellitus Tipo 2,172,8.3,138,88,29.8,1.3,201,Fundo de olho
3,PAC004,39,M,Diabetes Mellitus Tipo 2,96,5.9,122,78,25.6,0.8,174,None
4,PAC005,58,F,Diabetes Mellitus Tipo 2,238,10.1,154,98,33.5,1.5,243,Avaliação renal


In [ ]:
print("Quantidade de pacientes:", len(df_pacientes))
print()
print("Colunas:")
print(df_pacientes.columns.tolist())
print()
print("Tipos:")
print(df_pacientes.dtypes)

Quantidade de pacientes: 5

Colunas:
['patient_id', 'idade', 'sexo', 'diagnostico', 'glicemia_mg_dl', 'hba1c_percentual', 'pressao_sistolica', 'pressao_diastolica', 'imc', 'creatinina_mg_dl', 'colesterol_total_mg_dl', 'exame_pendente']

Tipos:
patient_id                 object
idade                       int64
sexo                       object
diagnostico                object
glicemia_mg_dl              int64
hba1c_percentual          float64
pressao_sistolica           int64
pressao_diastolica          int64
imc                       float64
creatinina_mg_dl          float64
colesterol_total_mg_dl      int64
exame_pendente             object
dtype: object


In [ ]:
df_pacientes.isnull().sum()

,0
patient_id,0
idade,0
sexo,0
diagnostico,0
glicemia_mg_dl,0
hba1c_percentual,0
pressao_sistolica,0
pressao_diastolica,0
imc,0
creatinina_mg_dl,0


In [ ]:
arquivo_pacientes = SYNTHETIC_DIR / "pacientes_sinteticos.csv"

df_pacientes.to_csv(
    arquivo_pacientes,
    index=False,
    encoding="utf-8"
)

print(f"Arquivo salvo em: {arquivo_pacientes}")

Arquivo salvo em: /content/fiap-tech-challenge-fase-3/data/synthetic/pacientes_sinteticos.csv


## 2 Anonimização e proteção de dados

Embora os dados utilizados neste projeto sejam sintéticos, esta etapa demonstra um pipeline de anonimização aplicável a dados clínicos antes de sua utilização por modelos de linguagem.

Para a demonstração, serão adicionados dados pessoais fictícios aos registros sintéticos. Em seguida, informações pessoalmente identificáveis (PII), como nome, CPF, e-mail e telefone, serão removidas ou mascaradas.

O objetivo é garantir que apenas informações clínicas necessárias sejam mantidas para as etapas posteriores do projeto.

In [ ]:
dados_identificaveis = df_pacientes.copy()

dados_identificaveis["nome"] = [
    "Ana Souza",
    "Carlos Lima",
    "Mariana Santos",
    "João Oliveira",
    "Fernanda Costa"
]

dados_identificaveis["cpf"] = [
    "111.222.333-44",
    "222.333.444-55",
    "333.444.555-66",
    "444.555.666-77",
    "555.666.777-88"
]

dados_identificaveis["email"] = [
    "ana.souza@email.com",
    "carlos.lima@email.com",
    "mariana.santos@email.com",
    "joao.oliveira@email.com",
    "fernanda.costa@email.com"
]

dados_identificaveis["telefone"] = [
    "(81) 99999-1001",
    "(81) 99999-1002",
    "(81) 99999-1003",
    "(81) 99999-1004",
    "(81) 99999-1005"
]

dados_identificaveis[
    ["patient_id", "nome", "cpf", "email", "telefone"]
]

,patient_id,nome,cpf,email,telefone
0,PAC001,Ana Souza,111.222.333-44,ana.souza@email.com,(81) 99999-1001
1,PAC002,Carlos Lima,222.333.444-55,carlos.lima@email.com,(81) 99999-1002
2,PAC003,Mariana Santos,333.444.555-66,mariana.santos@email.com,(81) 99999-1003
3,PAC004,João Oliveira,444.555.666-77,joao.oliveira@email.com,(81) 99999-1004
4,PAC005,Fernanda Costa,555.666.777-88,fernanda.costa@email.com,(81) 99999-1005


In [ ]:
COLUNAS_PII = [
    "nome",
    "cpf",
    "email",
    "telefone"
]

def anonimizar_dados(df):
    df_anonimizado = df.copy()

    colunas_encontradas = [
        coluna
        for coluna in COLUNAS_PII
        if coluna in df_anonimizado.columns
    ]

    df_anonimizado = df_anonimizado.drop(
        columns=colunas_encontradas
    )

    return df_anonimizado


df_anonimizado = anonimizar_dados(dados_identificaveis)

df_anonimizado.head()

,patient_id,idade,sexo,diagnostico,glicemia_mg_dl,hba1c_percentual,pressao_sistolica,pressao_diastolica,imc,creatinina_mg_dl,colesterol_total_mg_dl,exame_pendente
0,PAC001,52,F,Diabetes Mellitus Tipo 2,205,9.2,145,95,31.2,1.0,218,Microalbuminúria
1,PAC002,46,M,Diabetes Mellitus Tipo 2,118,6.7,128,82,27.4,0.9,185,None
2,PAC003,63,F,Diabetes Mellitus Tipo 2,172,8.3,138,88,29.8,1.3,201,Fundo de olho
3,PAC004,39,M,Diabetes Mellitus Tipo 2,96,5.9,122,78,25.6,0.8,174,None
4,PAC005,58,F,Diabetes Mellitus Tipo 2,238,10.1,154,98,33.5,1.5,243,Avaliação renal


In [ ]:
pii_restante = [
    coluna
    for coluna in COLUNAS_PII
    if coluna in df_anonimizado.columns
]

if len(pii_restante) == 0:
    print("Anonimização concluída com sucesso.")
    print("Nenhuma coluna de PII permanece no dataset.")
else:
    print("Atenção: ainda existem colunas de PII:")
    print(pii_restante)

Anonimização concluída com sucesso.
Nenhuma coluna de PII permanece no dataset.


## 3 Preprocessing e normalização

Após a anonimização, os registros clínicos são submetidos a uma etapa de preprocessing para melhorar sua consistência e qualidade.

Serão realizadas as seguintes operações:

- tratamento de valores ausentes;
- padronização de campos textuais;
- remoção de espaços desnecessários;
- verificação de duplicidades;
- validação dos tipos de dados;
- validação de intervalos clínicos;
- preparação do dataset processado.

In [ ]:
df_processado = df_anonimizado.copy()

df_processado["exame_pendente"] = (
    df_processado["exame_pendente"]
    .fillna("Sem exame pendente")
)

df_processado[["patient_id", "exame_pendente"]]

,patient_id,exame_pendente
0,PAC001,Microalbuminúria
1,PAC002,Sem exame pendente
2,PAC003,Fundo de olho
3,PAC004,Sem exame pendente
4,PAC005,Avaliação renal


In [ ]:
colunas_texto = [
    "sexo",
    "diagnostico",
    "exame_pendente"
]

for coluna in colunas_texto:
    df_processado[coluna] = (
        df_processado[coluna]
        .astype(str)
        .str.strip()
    )

df_processado.head()

,patient_id,idade,sexo,diagnostico,glicemia_mg_dl,hba1c_percentual,pressao_sistolica,pressao_diastolica,imc,creatinina_mg_dl,colesterol_total_mg_dl,exame_pendente
0,PAC001,52,F,Diabetes Mellitus Tipo 2,205,9.2,145,95,31.2,1.0,218,Microalbuminúria
1,PAC002,46,M,Diabetes Mellitus Tipo 2,118,6.7,128,82,27.4,0.9,185,Sem exame pendente
2,PAC003,63,F,Diabetes Mellitus Tipo 2,172,8.3,138,88,29.8,1.3,201,Fundo de olho
3,PAC004,39,M,Diabetes Mellitus Tipo 2,96,5.9,122,78,25.6,0.8,174,Sem exame pendente
4,PAC005,58,F,Diabetes Mellitus Tipo 2,238,10.1,154,98,33.5,1.5,243,Avaliação renal


In [ ]:
quantidade_duplicados = df_processado.duplicated().sum()

print("Registros duplicados:", quantidade_duplicados)

Registros duplicados: 0


In [ ]:
df_processado = (
    df_processado
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Quantidade final de registros:", len(df_processado))

Quantidade final de registros: 5


In [ ]:
valores_ausentes = df_processado.isnull().sum()

print("Valores ausentes após preprocessing:")
print(valores_ausentes)

Valores ausentes após preprocessing:
patient_id                0
idade                     0
sexo                      0
diagnostico               0
glicemia_mg_dl            0
hba1c_percentual          0
pressao_sistolica         0
pressao_diastolica        0
imc                       0
creatinina_mg_dl          0
colesterol_total_mg_dl    0
exame_pendente            0
dtype: int64


In [ ]:
validacoes = {
    "idade_valida": df_processado["idade"].between(0, 120),
    "glicemia_valida": df_processado["glicemia_mg_dl"].between(20, 1000),
    "hba1c_valida": df_processado["hba1c_percentual"].between(2, 20),
    "pressao_sistolica_valida": df_processado["pressao_sistolica"].between(50, 300),
    "pressao_diastolica_valida": df_processado["pressao_diastolica"].between(30, 200),
    "imc_valido": df_processado["imc"].between(10, 80),
    "creatinina_valida": df_processado["creatinina_mg_dl"].between(0.1, 20)
}

resultado_validacao = pd.DataFrame(validacoes)

resultado_validacao

,idade_valida,glicemia_valida,hba1c_valida,pressao_sistolica_valida,pressao_diastolica_valida,imc_valido,creatinina_valida
0,True,True,True,True,True,True,True
1,True,True,True,True,True,True,True
2,True,True,True,True,True,True,True
3,True,True,True,True,True,True,True
4,True,True,True,True,True,True,True


In [ ]:
registros_validos = resultado_validacao.all(axis=1)

print("Registros válidos:", registros_validos.sum())
print("Registros inválidos:", (~registros_validos).sum())

Registros válidos: 5
Registros inválidos: 0


## 4 Curadoria dos dados

A etapa de curadoria garante que somente registros consistentes sejam utilizados nas próximas fases do projeto.

Foram considerados os seguintes critérios:

- ausência de informações pessoais identificáveis;
- ausência de registros duplicados;
- tratamento de valores ausentes;
- consistência dos tipos de dados;
- plausibilidade dos valores numéricos;
- manutenção apenas das informações necessárias ao contexto clínico.

Registros que não atendam às regras de validação são excluídos do conjunto curado.

In [ ]:
df_curado = (
    df_processado.loc[registros_validos]
    .reset_index(drop=True)
)

print("Registros antes da curadoria:", len(df_processado))
print("Registros após a curadoria:", len(df_curado))

df_curado

Registros antes da curadoria: 5
Registros após a curadoria: 5


,patient_id,idade,sexo,diagnostico,glicemia_mg_dl,hba1c_percentual,pressao_sistolica,pressao_diastolica,imc,creatinina_mg_dl,colesterol_total_mg_dl,exame_pendente
0,PAC001,52,F,Diabetes Mellitus Tipo 2,205,9.2,145,95,31.2,1.0,218,Microalbuminúria
1,PAC002,46,M,Diabetes Mellitus Tipo 2,118,6.7,128,82,27.4,0.9,185,Sem exame pendente
2,PAC003,63,F,Diabetes Mellitus Tipo 2,172,8.3,138,88,29.8,1.3,201,Fundo de olho
3,PAC004,39,M,Diabetes Mellitus Tipo 2,96,5.9,122,78,25.6,0.8,174,Sem exame pendente
4,PAC005,58,F,Diabetes Mellitus Tipo 2,238,10.1,154,98,33.5,1.5,243,Avaliação renal


In [ ]:
arquivo_processado = PROCESSED_DIR / "pacientes_processados.csv"

df_curado.to_csv(
    arquivo_processado,
    index=False,
    encoding="utf-8"
)

print(f"Dataset processado salvo em: {arquivo_processado}")

Dataset processado salvo em: /content/fiap-tech-challenge-fase-3/data/processed/pacientes_processados.csv


In [ ]:
resumo_preprocessing = {
    "registros_iniciais": len(dados_identificaveis),
    "campos_pii_removidos": len(COLUNAS_PII),
    "registros_duplicados_identificados": int(quantidade_duplicados),
    "registros_invalidos": int((~registros_validos).sum()),
    "registros_finais": len(df_curado),
    "valores_ausentes_finais": int(df_curado.isnull().sum().sum())
}

pd.DataFrame(
    resumo_preprocessing.items(),
    columns=["Métrica", "Resultado"]
)

,Métrica,Resultado
0,registros_iniciais,5
1,campos_pii_removidos,4
2,registros_duplicados_identificados,0
3,registros_invalidos,0
4,registros_finais,5
5,valores_ausentes_finais,0


## 5 Preparação do dataset para fine-tuning

Após a etapa de anonimização, preprocessing e curadoria, será criado um conjunto de exemplos para o fine-tuning da LLM.

O dataset será composto por exemplos sintéticos relacionados a:

- protocolos clínicos institucionais;
- perguntas frequentes realizadas por médicos;
- interpretação de dados clínicos;
- exames pendentes;
- alertas;
- limites de atuação do assistente.

Cada exemplo será estruturado no formato:

- `instruction`: instrução ou pergunta realizada ao modelo;
- `input`: contexto clínico complementar;
- `output`: resposta esperada do assistente.

As respostas serão formuladas como apoio à decisão clínica e não substituirão a avaliação do profissional responsável.

In [ ]:
protocolos = [
    {
        "protocolo_id": "PROTO-DM-001",
        "titulo": "Monitoramento do controle glicêmico",
        "conteudo": (
            "Pacientes com Diabetes Mellitus Tipo 2 devem ter o controle "
            "glicêmico acompanhado periodicamente por meio de avaliação clínica "
            "e exames laboratoriais. Alterações persistentes devem ser analisadas "
            "pelo médico responsável considerando o histórico individual."
        )
    },
    {
        "protocolo_id": "PROTO-DM-002",
        "titulo": "Avaliação renal",
        "conteudo": (
            "Pacientes com Diabetes Mellitus Tipo 2 devem ser acompanhados "
            "quanto à função renal. Exames como creatinina e avaliação de "
            "albuminúria podem fazer parte do acompanhamento conforme avaliação médica."
        )
    },
    {
        "protocolo_id": "PROTO-DM-003",
        "titulo": "Avaliação oftalmológica",
        "conteudo": (
            "O acompanhamento de pessoas com Diabetes Mellitus Tipo 2 pode incluir "
            "avaliação oftalmológica periódica para rastreamento de alterações "
            "relacionadas à doença."
        )
    },
    {
        "protocolo_id": "PROTO-DM-004",
        "titulo": "Segurança do assistente clínico",
        "conteudo": (
            "O assistente virtual deve atuar exclusivamente como ferramenta de "
            "apoio à decisão clínica. Não deve emitir diagnóstico definitivo, "
            "prescrever medicamentos ou substituir a validação médica."
        )
    }
]

df_protocolos = pd.DataFrame(protocolos)

df_protocolos

,protocolo_id,titulo,conteudo
0,PROTO-DM-001,Monitoramento do controle glicêmico,Pacientes com Diabetes Mellitus Tipo 2 devem t...
1,PROTO-DM-002,Avaliação renal,Pacientes com Diabetes Mellitus Tipo 2 devem s...
2,PROTO-DM-003,Avaliação oftalmológica,O acompanhamento de pessoas com Diabetes Melli...
3,PROTO-DM-004,Segurança do assistente clínico,O assistente virtual deve atuar exclusivamente...


In [ ]:
arquivo_protocolos = SYNTHETIC_DIR / "protocolos_sinteticos.csv"

df_protocolos.to_csv(
    arquivo_protocolos,
    index=False,
    encoding="utf-8"
)

print(f"Protocolos salvos em: {arquivo_protocolos}")

Protocolos salvos em: /content/fiap-tech-challenge-fase-3/data/synthetic/protocolos_sinteticos.csv


In [ ]:
faqs = [
    {
        "instruction": "Quais dados devem ser observados no acompanhamento de um paciente com Diabetes Mellitus Tipo 2?",
        "input": "",
        "output": (
            "O acompanhamento pode considerar informações como glicemia, HbA1c, "
            "pressão arterial, função renal, perfil lipídico, IMC e exames pendentes. "
            "A interpretação deve considerar o histórico individual e ser validada "
            "pelo médico responsável."
        )
    },
    {
        "instruction": "O assistente pode prescrever medicamentos para o paciente?",
        "input": "",
        "output": (
            "Não. O assistente atua apenas como ferramenta de apoio à decisão clínica. "
            "Prescrições e alterações terapêuticas devem ser realizadas e validadas "
            "por profissional médico."
        )
    },
    {
        "instruction": "Como o assistente deve agir ao identificar um exame pendente?",
        "input": "",
        "output": (
            "O assistente pode sinalizar a existência do exame pendente e apresentar "
            "o protocolo institucional relacionado, mas a decisão sobre solicitação, "
            "prioridade ou conduta deve ser validada pelo médico responsável."
        )
    }
]

df_faqs = pd.DataFrame(faqs)

df_faqs

,instruction,input,output
0,Quais dados devem ser observados no acompanham...,,O acompanhamento pode considerar informações c...
1,O assistente pode prescrever medicamentos para...,,Não. O assistente atua apenas como ferramenta ...
2,Como o assistente deve agir ao identificar um ...,,O assistente pode sinalizar a existência do ex...


In [ ]:
def criar_contexto_paciente(row):
    return (
        f"Paciente {row['patient_id']}, {row['idade']} anos, sexo {row['sexo']}. "
        f"Diagnóstico: {row['diagnostico']}. "
        f"Glicemia: {row['glicemia_mg_dl']} mg/dL. "
        f"HbA1c: {row['hba1c_percentual']}%. "
        f"Pressão arterial: {row['pressao_sistolica']}/"
        f"{row['pressao_diastolica']} mmHg. "
        f"IMC: {row['imc']}. "
        f"Creatinina: {row['creatinina_mg_dl']} mg/dL. "
        f"Colesterol total: {row['colesterol_total_mg_dl']} mg/dL. "
        f"Exame pendente: {row['exame_pendente']}."
    )


df_curado["contexto_clinico"] = df_curado.apply(
    criar_contexto_paciente,
    axis=1
)

df_curado[
    ["patient_id", "contexto_clinico"]
]

,patient_id,contexto_clinico
0,PAC001,"Paciente PAC001, 52 anos, sexo F. Diagnóstico:..."
1,PAC002,"Paciente PAC002, 46 anos, sexo M. Diagnóstico:..."
2,PAC003,"Paciente PAC003, 63 anos, sexo F. Diagnóstico:..."
3,PAC004,"Paciente PAC004, 39 anos, sexo M. Diagnóstico:..."
4,PAC005,"Paciente PAC005, 58 anos, sexo F. Diagnóstico:..."


In [ ]:
exemplos_clinicos = []

for _, paciente in df_curado.iterrows():
    exemplos_clinicos.append(
        {
            "instruction": "Resuma os principais dados clínicos deste paciente.",
            "input": paciente["contexto_clinico"],
            "output": (
                f"O paciente {paciente['patient_id']} possui diagnóstico de "
                f"{paciente['diagnostico']}. "
                f"Apresenta glicemia de {paciente['glicemia_mg_dl']} mg/dL, "
                f"HbA1c de {paciente['hba1c_percentual']}%, "
                f"pressão arterial de {paciente['pressao_sistolica']}/"
                f"{paciente['pressao_diastolica']} mmHg e "
                f"exame pendente: {paciente['exame_pendente']}. "
                "Essas informações devem ser interpretadas em conjunto com o "
                "histórico clínico e validadas pelo médico responsável."
            )
        }
    )

df_exemplos_clinicos = pd.DataFrame(exemplos_clinicos)

df_exemplos_clinicos

,instruction,input,output
0,Resuma os principais dados clínicos deste paci...,"Paciente PAC001, 52 anos, sexo F. Diagnóstico:...",O paciente PAC001 possui diagnóstico de Diabet...
1,Resuma os principais dados clínicos deste paci...,"Paciente PAC002, 46 anos, sexo M. Diagnóstico:...",O paciente PAC002 possui diagnóstico de Diabet...
2,Resuma os principais dados clínicos deste paci...,"Paciente PAC003, 63 anos, sexo F. Diagnóstico:...",O paciente PAC003 possui diagnóstico de Diabet...
3,Resuma os principais dados clínicos deste paci...,"Paciente PAC004, 39 anos, sexo M. Diagnóstico:...",O paciente PAC004 possui diagnóstico de Diabet...
4,Resuma os principais dados clínicos deste paci...,"Paciente PAC005, 58 anos, sexo F. Diagnóstico:...",O paciente PAC005 possui diagnóstico de Diabet...


In [ ]:
exemplos_protocolos = []

for _, protocolo in df_protocolos.iterrows():
    exemplos_protocolos.append(
        {
            "instruction": f"Explique o protocolo: {protocolo['titulo']}.",
            "input": "",
            "output": protocolo["conteudo"]
        }
    )

df_exemplos_protocolos = pd.DataFrame(exemplos_protocolos)

df_exemplos_protocolos

,instruction,input,output
0,Explique o protocolo: Monitoramento do control...,,Pacientes com Diabetes Mellitus Tipo 2 devem t...
1,Explique o protocolo: Avaliação renal.,,Pacientes com Diabetes Mellitus Tipo 2 devem s...
2,Explique o protocolo: Avaliação oftalmológica.,,O acompanhamento de pessoas com Diabetes Melli...
3,Explique o protocolo: Segurança do assistente ...,,O assistente virtual deve atuar exclusivamente...


In [ ]:
df_finetuning = pd.concat(
    [
        df_faqs,
        df_exemplos_protocolos,
        df_exemplos_clinicos
    ],
    ignore_index=True
)

print("Quantidade total de exemplos:", len(df_finetuning))

df_finetuning

Quantidade total de exemplos: 12


,instruction,input,output
0,Quais dados devem ser observados no acompanham...,,O acompanhamento pode considerar informações c...
1,O assistente pode prescrever medicamentos para...,,Não. O assistente atua apenas como ferramenta ...
2,Como o assistente deve agir ao identificar um ...,,O assistente pode sinalizar a existência do ex...
3,Explique o protocolo: Monitoramento do control...,,Pacientes com Diabetes Mellitus Tipo 2 devem t...
4,Explique o protocolo: Avaliação renal.,,Pacientes com Diabetes Mellitus Tipo 2 devem s...
5,Explique o protocolo: Avaliação oftalmológica.,,O acompanhamento de pessoas com Diabetes Melli...
6,Explique o protocolo: Segurança do assistente ...,,O assistente virtual deve atuar exclusivamente...
7,Resuma os principais dados clínicos deste paci...,"Paciente PAC001, 52 anos, sexo F. Diagnóstico:...",O paciente PAC001 possui diagnóstico de Diabet...
8,Resuma os principais dados clínicos deste paci...,"Paciente PAC002, 46 anos, sexo M. Diagnóstico:...",O paciente PAC002 possui diagnóstico de Diabet...
9,Resuma os principais dados clínicos deste paci...,"Paciente PAC003, 63 anos, sexo F. Diagnóstico:...",O paciente PAC003 possui diagnóstico de Diabet...


In [ ]:
print("Valores ausentes:")
print(df_finetuning.isnull().sum())

print()
print("Duplicados:", df_finetuning.duplicated().sum())

print()
print("Colunas:", df_finetuning.columns.tolist())

Valores ausentes:
instruction    0
input          0
output         0
dtype: int64

Duplicados: 0

Colunas: ['instruction', 'input', 'output']


In [ ]:
df_finetuning = (
    df_finetuning
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Quantidade após curadoria:", len(df_finetuning))

Quantidade após curadoria: 12


In [ ]:
arquivo_finetuning = PROCESSED_DIR / "dataset_finetuning.jsonl"

df_finetuning.to_json(
    arquivo_finetuning,
    orient="records",
    lines=True,
    force_ascii=False
)

print(f"Dataset de fine-tuning salvo em: {arquivo_finetuning}")

Dataset de fine-tuning salvo em: /content/fiap-tech-challenge-fase-3/data/processed/dataset_finetuning.jsonl


In [ ]:
with open(
    arquivo_finetuning,
    "r",
    encoding="utf-8"
) as arquivo:
    for indice, linha in enumerate(arquivo):
        print(linha.strip())

        if indice == 2:
            break

{"instruction":"Quais dados devem ser observados no acompanhamento de um paciente com Diabetes Mellitus Tipo 2?","input":"","output":"O acompanhamento pode considerar informações como glicemia, HbA1c, pressão arterial, função renal, perfil lipídico, IMC e exames pendentes. A interpretação deve considerar o histórico individual e ser validada pelo médico responsável."}
{"instruction":"O assistente pode prescrever medicamentos para o paciente?","input":"","output":"Não. O assistente atua apenas como ferramenta de apoio à decisão clínica. Prescrições e alterações terapêuticas devem ser realizadas e validadas por profissional médico."}
{"instruction":"Como o assistente deve agir ao identificar um exame pendente?","input":"","output":"O assistente pode sinalizar a existência do exame pendente e apresentar o protocolo institucional relacionado, mas a decisão sobre solicitação, prioridade ou conduta deve ser validada pelo médico responsável."}


In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_validation = train_test_split(
    df_finetuning,
    test_size=0.2,
    random_state=SEED
)

print("Treino:", len(df_train))
print("Validação:", len(df_validation))

Treino: 9
Validação: 3


In [ ]:
arquivo_train = PROCESSED_DIR / "train.jsonl"
arquivo_validation = PROCESSED_DIR / "validation.jsonl"

df_train.to_json(
    arquivo_train,
    orient="records",
    lines=True,
    force_ascii=False
)

df_validation.to_json(
    arquivo_validation,
    orient="records",
    lines=True,
    force_ascii=False
)

print(f"Treino salvo em: {arquivo_train}")
print(f"Validação salva em: {arquivo_validation}")

Treino salvo em: /content/fiap-tech-challenge-fase-3/data/processed/train.jsonl
Validação salva em: /content/fiap-tech-challenge-fase-3/data/processed/validation.jsonl


## 6 Resultado da preparação dos dados

Ao final desta etapa foram produzidos:

- dataset sintético de pacientes;
- pipeline demonstrável de anonimização de PII;
- dataset clínico normalizado e curado;
- protocolos institucionais sintéticos;
- perguntas frequentes médicas;
- exemplos clínicos contextualizados;
- dataset estruturado para fine-tuning;
- conjuntos separados de treino e validação.

Os arquivos gerados serão utilizados nas próximas etapas para o fine-tuning da LLM e para a construção do assistente médico com LangChain e LangGraph.

## 7 Ampliação do dataset de treinamento

Para aumentar a diversidade do conjunto de treinamento, serão criadas variações sintéticas das instruções existentes.

As variações preservam o significado clínico e as regras de segurança, mas utilizam diferentes formas de pergunta e contextualização.

O objetivo é reduzir a dependência de um único padrão textual e melhorar a capacidade do modelo de responder a diferentes formulações.

In [ ]:
variacoes_faq = [
    {
        "instruction": "Quais informações clínicas são relevantes no acompanhamento de diabetes tipo 2?",
        "input": "",
        "output": df_faqs.iloc[0]["output"]
    },
    {
        "instruction": "Que indicadores devem ser avaliados em um paciente com diabetes tipo 2?",
        "input": "",
        "output": df_faqs.iloc[0]["output"]
    },
    {
        "instruction": "Quais exames e dados devem ser acompanhados em pacientes com diabetes tipo 2?",
        "input": "",
        "output": df_faqs.iloc[0]["output"]
    },
    {
        "instruction": "O assistente está autorizado a prescrever um medicamento?",
        "input": "",
        "output": df_faqs.iloc[1]["output"]
    },
    {
        "instruction": "A IA pode definir ou alterar a medicação do paciente?",
        "input": "",
        "output": df_faqs.iloc[1]["output"]
    },
    {
        "instruction": "O assistente pode realizar uma prescrição sem validação médica?",
        "input": "",
        "output": df_faqs.iloc[1]["output"]
    },
    {
        "instruction": "O que fazer quando existe um exame pendente no prontuário?",
        "input": "",
        "output": df_faqs.iloc[2]["output"]
    },
    {
        "instruction": "Como sinalizar exames que ainda não foram realizados?",
        "input": "",
        "output": df_faqs.iloc[2]["output"]
    },
    {
        "instruction": "Como o sistema deve lidar com exames pendentes?",
        "input": "",
        "output": df_faqs.iloc[2]["output"]
    }
]

df_variacoes_faq = pd.DataFrame(variacoes_faq)

df_variacoes_faq

,instruction,input,output
0,Quais informações clínicas são relevantes no a...,,O acompanhamento pode considerar informações c...
1,Que indicadores devem ser avaliados em um paci...,,O acompanhamento pode considerar informações c...
2,Quais exames e dados devem ser acompanhados em...,,O acompanhamento pode considerar informações c...
3,O assistente está autorizado a prescrever um m...,,Não. O assistente atua apenas como ferramenta ...
4,A IA pode definir ou alterar a medicação do pa...,,Não. O assistente atua apenas como ferramenta ...
5,O assistente pode realizar uma prescrição sem ...,,Não. O assistente atua apenas como ferramenta ...
6,O que fazer quando existe um exame pendente no...,,O assistente pode sinalizar a existência do ex...
7,Como sinalizar exames que ainda não foram real...,,O assistente pode sinalizar a existência do ex...
8,Como o sistema deve lidar com exames pendentes?,,O assistente pode sinalizar a existência do ex...


In [ ]:
variacoes_pacientes = []

instrucoes_clinicas = [
    "Resuma o quadro clínico deste paciente.",
    "Apresente os principais dados clínicos registrados.",
    "Sintetize as informações clínicas deste prontuário.",
    "Liste os principais achados clínicos deste paciente.",
    "Faça um resumo objetivo dos dados deste paciente."
]

for _, paciente in df_curado.iterrows():
    for instrucao in instrucoes_clinicas:
        variacoes_pacientes.append(
            {
                "instruction": instrucao,
                "input": paciente["contexto_clinico"],
                "output": (
                    f"O paciente {paciente['patient_id']} possui diagnóstico de "
                    f"{paciente['diagnostico']}. "
                    f"Apresenta glicemia de {paciente['glicemia_mg_dl']} mg/dL, "
                    f"HbA1c de {paciente['hba1c_percentual']}%, "
                    f"pressão arterial de {paciente['pressao_sistolica']}/"
                    f"{paciente['pressao_diastolica']} mmHg e "
                    f"exame pendente: {paciente['exame_pendente']}. "
                    "As informações devem ser avaliadas em conjunto com o histórico "
                    "clínico e validadas pelo médico responsável."
                )
            }
        )

df_variacoes_pacientes = pd.DataFrame(variacoes_pacientes)

print("Novos exemplos clínicos:", len(df_variacoes_pacientes))

df_variacoes_pacientes.head()

Novos exemplos clínicos: 25


,instruction,input,output
0,Resuma o quadro clínico deste paciente.,"Paciente PAC001, 52 anos, sexo F. Diagnóstico:...",O paciente PAC001 possui diagnóstico de Diabet...
1,Apresente os principais dados clínicos registr...,"Paciente PAC001, 52 anos, sexo F. Diagnóstico:...",O paciente PAC001 possui diagnóstico de Diabet...
2,Sintetize as informações clínicas deste prontu...,"Paciente PAC001, 52 anos, sexo F. Diagnóstico:...",O paciente PAC001 possui diagnóstico de Diabet...
3,Liste os principais achados clínicos deste pac...,"Paciente PAC001, 52 anos, sexo F. Diagnóstico:...",O paciente PAC001 possui diagnóstico de Diabet...
4,Faça um resumo objetivo dos dados deste paciente.,"Paciente PAC001, 52 anos, sexo F. Diagnóstico:...",O paciente PAC001 possui diagnóstico de Diabet...


In [ ]:
variacoes_protocolos = []

instrucoes_protocolos = [
    "Explique o protocolo institucional sobre",
    "Resuma o protocolo institucional sobre",
    "Apresente as orientações do protocolo sobre"
]

for _, protocolo in df_protocolos.iterrows():
    for prefixo in instrucoes_protocolos:
        variacoes_protocolos.append(
            {
                "instruction": f"{prefixo} {protocolo['titulo']}.",
                "input": "",
                "output": protocolo["conteudo"]
            }
        )

df_variacoes_protocolos = pd.DataFrame(variacoes_protocolos)

print("Novos exemplos de protocolos:", len(df_variacoes_protocolos))

df_variacoes_protocolos.head()

Novos exemplos de protocolos: 12


,instruction,input,output
0,Explique o protocolo institucional sobre Monit...,,Pacientes com Diabetes Mellitus Tipo 2 devem t...
1,Resuma o protocolo institucional sobre Monitor...,,Pacientes com Diabetes Mellitus Tipo 2 devem t...
2,Apresente as orientações do protocolo sobre Mo...,,Pacientes com Diabetes Mellitus Tipo 2 devem t...
3,Explique o protocolo institucional sobre Avali...,,Pacientes com Diabetes Mellitus Tipo 2 devem s...
4,Resuma o protocolo institucional sobre Avaliaç...,,Pacientes com Diabetes Mellitus Tipo 2 devem s...


In [ ]:
df_finetuning_ampliado = pd.concat(
    [
        df_finetuning,
        df_variacoes_faq,
        df_variacoes_pacientes,
        df_variacoes_protocolos
    ],
    ignore_index=True
)

df_finetuning_ampliado = (
    df_finetuning_ampliado
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Quantidade total após ampliação:", len(df_finetuning_ampliado))

df_finetuning_ampliado.head()

Quantidade total após ampliação: 58


,instruction,input,output
0,Quais dados devem ser observados no acompanham...,,O acompanhamento pode considerar informações c...
1,O assistente pode prescrever medicamentos para...,,Não. O assistente atua apenas como ferramenta ...
2,Como o assistente deve agir ao identificar um ...,,O assistente pode sinalizar a existência do ex...
3,Explique o protocolo: Monitoramento do control...,,Pacientes com Diabetes Mellitus Tipo 2 devem t...
4,Explique o protocolo: Avaliação renal.,,Pacientes com Diabetes Mellitus Tipo 2 devem s...


In [ ]:
print("Valores ausentes:")
print(df_finetuning_ampliado.isnull().sum())

print()
print("Duplicados:", df_finetuning_ampliado.duplicated().sum())

print()
print("Total:", len(df_finetuning_ampliado))

Valores ausentes:
instruction    0
input          0
output         0
dtype: int64

Duplicados: 0

Total: 58


In [ ]:
arquivo_finetuning_final = PROCESSED_DIR / "dataset_finetuning_final.jsonl"

df_finetuning_ampliado.to_json(
    arquivo_finetuning_final,
    orient="records",
    lines=True,
    force_ascii=False
)

print(f"Dataset final salvo em: {arquivo_finetuning_final}")

Dataset final salvo em: /content/fiap-tech-challenge-fase-3/data/processed/dataset_finetuning_final.jsonl


In [ ]:
df_train_final, df_validation_final = train_test_split(
    df_finetuning_ampliado,
    test_size=0.2,
    random_state=SEED,
    shuffle=True
)

print("Treino:", len(df_train_final))
print("Validação:", len(df_validation_final))

Treino: 46
Validação: 12


In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/FIAP/TechChallenge_Fase3"
)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FULL_DATASET_PATH = (
    PROCESSED_DIR / "dataset_finetuning_final.jsonl"
)

TRAIN_PATH = (
    PROCESSED_DIR / "train_final.jsonl"
)

VALIDATION_PATH = (
    PROCESSED_DIR / "validation_final.jsonl"
)


df_finetuning_ampliado.to_json(
    FULL_DATASET_PATH,
    orient="records",
    lines=True,
    force_ascii=False
)

df_train_final.to_json(
    TRAIN_PATH,
    orient="records",
    lines=True,
    force_ascii=False
)

df_validation_final.to_json(
    VALIDATION_PATH,
    orient="records",
    lines=True,
    force_ascii=False
)


print(
    "Dataset completo:",
    FULL_DATASET_PATH.exists(),
    "-",
    len(df_finetuning_ampliado),
    "registros"
)

print(
    "Treino:",
    TRAIN_PATH.exists(),
    "-",
    len(df_train_final),
    "registros"
)

print(
    "Validação:",
    VALIDATION_PATH.exists(),
    "-",
    len(df_validation_final),
    "registros"
)

Dataset completo: True - 58 registros
Treino: True - 46 registros
Validação: True - 12 registros


In [ ]:
arquivo_train_final = PROCESSED_DIR / "train_final.jsonl"
arquivo_validation_final = PROCESSED_DIR / "validation_final.jsonl"

df_train_final.to_json(
    arquivo_train_final,
    orient="records",
    lines=True,
    force_ascii=False
)

df_validation_final.to_json(
    arquivo_validation_final,
    orient="records",
    lines=True,
    force_ascii=False
)

print(f"Treino final salvo em: {arquivo_train_final}")
print(f"Validação final salva em: {arquivo_validation_final}")

Treino final salvo em: /content/drive/MyDrive/FIAP/TechChallenge_Fase3/data/processed/train_final.jsonl
Validação final salva em: /content/drive/MyDrive/FIAP/TechChallenge_Fase3/data/processed/validation_final.jsonl


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

ValueError: Mountpoint must not already contain files

In [ ]:
from pathlib import Path
import shutil

DRIVE_PROJECT_DIR = Path(
    "/content/drive/MyDrive/FIAP/TechChallenge_Fase3"
)

DRIVE_DATA_DIR = DRIVE_PROJECT_DIR / "data" / "processed"

DRIVE_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(DRIVE_DATA_DIR)

/content/drive/MyDrive/FIAP/TechChallenge_Fase3/data/processed


In [ ]:
arquivos_para_salvar = [
    "pacientes_processados.csv",
    "dataset_finetuning_final.jsonl",
    "train_final.jsonl",
    "validation_final.jsonl"
]

for nome_arquivo in arquivos_para_salvar:
    origem = PROCESSED_DIR / nome_arquivo
    destino = DRIVE_DATA_DIR / nome_arquivo

    if origem.exists():
        shutil.copy(origem, destino)
        print(f"Salvo: {destino}")
    else:
        print(f"Não encontrado: {origem}")

Não encontrado: /content/drive/MyDrive/FIAP/TechChallenge_Fase3/data/processed/pacientes_processados.csv


SameFileError: PosixPath('/content/drive/MyDrive/FIAP/TechChallenge_Fase3/data/processed/dataset_finetuning_final.jsonl') and PosixPath('/content/drive/MyDrive/FIAP/TechChallenge_Fase3/data/processed/dataset_finetuning_final.jsonl') are the same file

In [ ]:
DRIVE_SYNTHETIC_DIR = (
    DRIVE_PROJECT_DIR / "data" / "synthetic"
)

DRIVE_SYNTHETIC_DIR.mkdir(
    parents=True,
    exist_ok=True
)

arquivo_protocolos_drive = (
    DRIVE_SYNTHETIC_DIR / "protocolos_sinteticos.csv"
)

shutil.copy(
    SYNTHETIC_DIR / "protocolos_sinteticos.csv",
    arquivo_protocolos_drive
)

print(arquivo_protocolos_drive)

/content/drive/MyDrive/FIAP/TechChallenge_Fase3/data/synthetic/protocolos_sinteticos.csv
